# figpanel - Scientific Figure Panel Detector

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Plottie/figpanel/blob/main/examples/demo.ipynb)
[![PyPI](https://img.shields.io/pypi/v/figpanel?color=blue)](https://pypi.org/project/figpanel/)
[![Plottie](https://img.shields.io/badge/Built%20by-Plottie-purple)](https://plottie.art)

Detect and extract individual panels from composite scientific figures using YOLOv12.

**Built by [Plottie](https://plottie.art)** - the scientific plot discovery platform.

## 1. Installation

In [ ]:
# Install figpanel with full pipeline support
!pip install -q figpanel[full]

# On Colab, install Tesseract for OCR
import subprocess, sys
if 'google.colab' in sys.modules:
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'tesseract-ocr'], capture_output=True)

## 2. Load a Sample Image

Replace the URL below with any composite scientific figure.

In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display

# Use a local image or download one
image_path = "sample_figure.png"

# If running locally with example images:
local_images = list(Path("images").glob("*.png")) + list(Path("images").glob("*.jpg"))
if local_images:
    image_path = str(local_images[0])
    print(f"Using local image: {image_path}")
else:
    print(f"Place a composite figure image at '{image_path}' and re-run this cell.")

if Path(image_path).exists():
    img = Image.open(image_path)
    print(f"Image size: {img.size}")
    display(img)

## 3. Basic Detection

`figpanel.detect()` returns raw bounding boxes for subplots and caption labels.

In [ ]:
import figpanel

# Detect subplots and captions
results = figpanel.detect(image_path)

print(f"Detected {len(results['subplots'])} subplots")
print(f"Detected {len(results['captions'])} captions")

# Each entry is (x1, y1, x2, y2, confidence)
for i, sub in enumerate(results['subplots']):
    print(f"  Subplot {i+1}: bbox={sub[:4]}, conf={sub[4]:.2f}")
for i, cap in enumerate(results['captions']):
    print(f"  Caption {i+1}: bbox={cap[:4]}, conf={cap[4]:.2f}")

## 4. Visualize Detections

Draw bounding boxes on the image: subplots in blue, captions in green.

In [ ]:
# Visualize detections
annotated = figpanel.visualize(image_path)
display(annotated)

## 5. Full Pipeline - Extract Panels

`figpanel.extract()` runs the complete pipeline: detect, OCR, match captions to subplots, crop, and deduplicate.

In [ ]:
# Extract panels (saves cropped images to output/)
panels = figpanel.extract(image_path, "output/")

print(f"Extracted {len(panels)} panels:\n")
for panel in panels:
    print(f"Panel '{panel.label}': bbox={panel.bbox}, confidence={panel.confidence:.2f}")
    display(panel.image)
    print()

## 6. Batch Processing

Process multiple figures at once.

In [ ]:
from pathlib import Path

# Process all images in a directory
input_dir = Path("images")
output_dir = Path("output")

if input_dir.exists():
    for img_path in sorted(input_dir.glob("*.png")):
        print(f"Processing {img_path.name}...")
        panels = figpanel.extract(img_path, output_dir, prefix=img_path.stem)
        print(f"  -> {len(panels)} panels extracted")
else:
    print(f"Create an '{input_dir}' directory with figure images to try batch processing.")

## 7. Advanced Usage

Customize detection thresholds and pipeline parameters.

In [ ]:
# Adjust confidence threshold for more/fewer detections
results_strict = figpanel.detect(image_path, conf=0.5)   # Higher = fewer, more confident
results_loose = figpanel.detect(image_path, conf=0.1)    # Lower = more detections

print(f"Strict (conf=0.5): {len(results_strict['subplots'])} subplots")
print(f"Loose  (conf=0.1): {len(results_loose['subplots'])} subplots")

# Full pipeline with custom parameters
panels = figpanel.extract(
    image_path,
    conf=0.3,           # Detection confidence
    iou=0.4,            # NMS IoU threshold
    padding=8,          # Extra padding around crops
    dedup_thresh=0.5,   # Stricter deduplication
)
print(f"\nCustom pipeline: {len(panels)} panels")

## About Plottie

**figpanel** is built and maintained by [Plottie](https://plottie.art) - the scientific plot discovery platform.

Plottie helps researchers explore, collect, and find inspiration from high-quality scientific plots across open-access literature.

**Links:**
- [plottie.art](https://plottie.art) - Browse curated scientific figures
- [GitHub](https://github.com/Plottie/figpanel) - Source code and issues
- [HuggingFace](https://huggingface.co/plottie/figpanel-yolov12) - Model card and weights
- [PyPI](https://pypi.org/project/figpanel/) - Package page

---

If you use figpanel in your research, please cite:

```bibtex
@software{figpanel,
  title = {figpanel: Scientific Figure Panel Detector},
  author = {Plottie},
  url = {https://github.com/Plottie/figpanel},
  version = {0.1.0},
  year = {2026},
  note = {Built by Plottie (https://plottie.art)}
}
```